# Typhoon OCR Colab Workflow (Fast Runtime)

This notebook is synced with the latest local Python workflow and tuned to run faster on Colab.

What changed:

- the heavy read/write work runs on local Colab storage under `/content`
- results and OCR caches are synced back to Google Drive after each run
- you can choose a speed profile instead of manually tuning every OCR setting

It still uses the same project scripts:

- `ocr_typhoon_to_csv.py`
- `build_submission_from_ocr.py`

Recommended setup:

1. Put the whole project folder in Google Drive.
2. Open this notebook in Colab.
3. Edit only the config cell.
4. Run the helper cell once.
5. Run the baseline cell if needed.
6. Rerun the refinement cell until no more docs are selected.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('typhoon_ocr') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'typhoon-ocr'])
    print('Installed typhoon-ocr')
else:
    print('typhoon-ocr is already available')

In [ ]:
from google.colab import drive
from getpass import getpass
import os

drive.mount('/content/drive')

if not os.environ.get('TYPHOON_OCR_API_KEY'):
    os.environ['TYPHOON_OCR_API_KEY'] = getpass('Typhoon OCR API key: ')

print('Drive mounted and API key is ready.')

In [ ]:
from pathlib import Path

# Drive source paths.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/New project')
DRIVE_IMAGES_DIR = Path('/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data/images')

# Fast local runtime.
USE_LOCAL_RUNTIME = True
LOCAL_RUNTIME_ROOT = Path('/content/ocr_fast_runtime')
COPY_IMAGES_TO_LOCAL = False
SYNC_EXISTING_OUTPUTS_TO_LOCAL = True
SYNC_EXISTING_CACHES_TO_LOCAL = True

# OCR speed presets. Start with 'balanced'.
SPEED_PROFILE = 'balanced'  # conservative | balanced | aggressive
SPEED_PRESETS = {
    'conservative': {'workers': 1, 'max_requests_per_minute': 4, 'sleep_seconds': 12.0, 'batch_limit': 10},
    'balanced': {'workers': 2, 'max_requests_per_minute': 8, 'sleep_seconds': 0.0, 'batch_limit': 12},
    'aggressive': {'workers': 3, 'max_requests_per_minute': 12, 'sleep_seconds': 0.0, 'batch_limit': 16},
}

if SPEED_PROFILE not in SPEED_PRESETS:
    raise ValueError(f'Unknown SPEED_PROFILE: {SPEED_PROFILE}')

speed = SPEED_PRESETS[SPEED_PROFILE]
WORKERS = speed['workers']
MAX_REQUESTS_PER_MINUTE = speed['max_requests_per_minute']
SLEEP_SECONDS = speed['sleep_seconds']
BATCH_LIMIT = speed['batch_limit']
RETRIES = 4

if USE_LOCAL_RUNTIME:
    PROJECT_DIR = LOCAL_RUNTIME_ROOT / 'project'
    OUTPUT_DIR = PROJECT_DIR / 'output'
    OCR_SCRIPT = PROJECT_DIR / 'ocr_typhoon_to_csv.py'
    BUILD_SCRIPT = PROJECT_DIR / 'build_submission_from_ocr.py'
    TEMPLATE_CSV = PROJECT_DIR / 'summission_template2.csv'
    IMAGES_DIR = (LOCAL_RUNTIME_ROOT / 'images') if COPY_IMAGES_TO_LOCAL else DRIVE_IMAGES_DIR
else:
    PROJECT_DIR = DRIVE_PROJECT_DIR
    OUTPUT_DIR = PROJECT_DIR / 'output'
    OCR_SCRIPT = DRIVE_PROJECT_DIR / 'ocr_typhoon_to_csv.py'
    BUILD_SCRIPT = DRIVE_PROJECT_DIR / 'build_submission_from_ocr.py'
    TEMPLATE_CSV = DRIVE_PROJECT_DIR / 'summission_template2.csv'
    IMAGES_DIR = DRIVE_IMAGES_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_PROJECT_DIR =', DRIVE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('IMAGES_DIR =', IMAGES_DIR)
print('USE_LOCAL_RUNTIME =', USE_LOCAL_RUNTIME)
print('COPY_IMAGES_TO_LOCAL =', COPY_IMAGES_TO_LOCAL)
print('SPEED_PROFILE =', SPEED_PROFILE)
print('WORKERS =', WORKERS)
print('MAX_REQUESTS_PER_MINUTE =', MAX_REQUESTS_PER_MINUTE)
print('BATCH_LIMIT =', BATCH_LIMIT)

In [ ]:
from __future__ import annotations

import csv
import re
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

PYTHON_BIN = sys.executable
BATCH_NUMBER_RE = re.compile(r'batch(\d+)')
RUNTIME_STATE = {'prepared': False, 'images_local': False}


def run_cmd(args: list[str], cwd: Path = PROJECT_DIR) -> None:
    printable = ' '.join(str(arg) for arg in args)
    print(f'$ {printable}')
    subprocess.run([str(arg) for arg in args], cwd=str(cwd), check=True)


def load_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))


def batch_sort_key(path: Path) -> tuple[int, str]:
    match = BATCH_NUMBER_RE.search(path.name)
    return (int(match.group(1)) if match else -1, path.name)


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def copy_file_if_needed(src: Path, dst: Path) -> bool:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        src_stat = src.stat()
        dst_stat = dst.stat()
        if src_stat.st_size == dst_stat.st_size and int(src_stat.st_mtime) <= int(dst_stat.st_mtime):
            return False
    shutil.copy2(src, dst)
    return True


def sync_tree(src: Path, dst: Path, label: str) -> int:
    if not src.exists():
        print(f'[INFO] Skip sync for {label}: missing {src}')
        return 0

    files = [path for path in src.rglob('*') if path.is_file()]
    copied = 0
    for src_file in files:
        dst_file = dst / src_file.relative_to(src)
        if copy_file_if_needed(src_file, dst_file):
            copied += 1

    print(f'[INFO] Synced {label}: {copied}/{len(files)} files')
    return copied


def prepare_runtime(copy_images: bool = False, force: bool = False) -> None:
    if not USE_LOCAL_RUNTIME:
        return

    if RUNTIME_STATE['prepared'] and (not copy_images or RUNTIME_STATE['images_local']) and not force:
        return

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for src, dst in [
        (DRIVE_PROJECT_DIR / 'ocr_typhoon_to_csv.py', OCR_SCRIPT),
        (DRIVE_PROJECT_DIR / 'build_submission_from_ocr.py', BUILD_SCRIPT),
        (DRIVE_PROJECT_DIR / 'summission_template2.csv', TEMPLATE_CSV),
    ]:
        if not src.exists():
            print(f'[WARN] Missing source file in Drive: {src}')
            continue
        copy_file_if_needed(src, dst)

    if SYNC_EXISTING_OUTPUTS_TO_LOCAL:
        sync_tree(DRIVE_PROJECT_DIR / 'output', OUTPUT_DIR, 'drive output -> local output')

    if SYNC_EXISTING_CACHES_TO_LOCAL:
        sync_tree(DRIVE_PROJECT_DIR / 'ocr_cache', PROJECT_DIR / 'ocr_cache', 'drive ocr_cache -> local ocr_cache')
        for cache_dir in sorted(DRIVE_PROJECT_DIR.glob('temp_constituency_page1*_cache'), key=batch_sort_key):
            sync_tree(cache_dir, PROJECT_DIR / cache_dir.name, f'drive {cache_dir.name} -> local')

    if copy_images:
        sync_tree(DRIVE_IMAGES_DIR, LOCAL_RUNTIME_ROOT / 'images', 'drive images -> local images')
        RUNTIME_STATE['images_local'] = True

    RUNTIME_STATE['prepared'] = True


def sync_runtime_back_to_drive() -> None:
    if not USE_LOCAL_RUNTIME:
        return

    sync_tree(OUTPUT_DIR, DRIVE_PROJECT_DIR / 'output', 'local output -> drive output')
    sync_tree(PROJECT_DIR / 'ocr_cache', DRIVE_PROJECT_DIR / 'ocr_cache', 'local ocr_cache -> drive ocr_cache')

    for cache_dir in sorted(PROJECT_DIR.glob('temp_constituency_page1*_cache'), key=batch_sort_key):
        sync_tree(cache_dir, DRIVE_PROJECT_DIR / cache_dir.name, f'local {cache_dir.name} -> drive')


def assert_required_paths(require_images: bool = True, copy_images: bool = False) -> None:
    if USE_LOCAL_RUNTIME:
        prepare_runtime(copy_images=copy_images)

    required = [
        ('PROJECT_DIR', PROJECT_DIR),
        ('OCR_SCRIPT', OCR_SCRIPT),
        ('BUILD_SCRIPT', BUILD_SCRIPT),
        ('TEMPLATE_CSV', TEMPLATE_CSV),
    ]
    if require_images:
        required.append(('IMAGES_DIR', IMAGES_DIR))

    missing = [name for name, path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            f'Missing required paths: {missing}. Put the files in {DRIVE_PROJECT_DIR} or update the config cell.'
        )


def find_page1_cache_dirs() -> list[Path]:
    cache_dirs = [PROJECT_DIR / 'ocr_cache']
    cache_dirs.extend(sorted(PROJECT_DIR.glob('temp_constituency_page1*_cache'), key=batch_sort_key))

    unique_dirs: list[Path] = []
    seen: set[Path] = set()
    for path in cache_dirs:
        normalized = path.resolve(strict=False)
        if normalized in seen:
            continue
        seen.add(normalized)
        unique_dirs.append(path)
    return unique_dirs


def detect_current_paths() -> tuple[Path | None, Path | None]:
    ocr_candidates = sorted(OUTPUT_DIR.glob('ocr_results_final_round_plus_page1_batch*.csv'), key=batch_sort_key)
    submission_candidates = sorted(OUTPUT_DIR.glob('submission_id_votes_batch*_auto_balance.csv'), key=batch_sort_key)

    current_ocr = ocr_candidates[-1] if ocr_candidates else None
    current_submission = submission_candidates[-1] if submission_candidates else None

    if current_ocr is None:
        current_ocr = first_existing([
            OUTPUT_DIR / 'ocr_results_targeted_plus_page1.csv',
            OUTPUT_DIR / 'ocr_results_refresh.csv',
        ])

    if current_submission is None:
        current_submission = first_existing([
            OUTPUT_DIR / 'submission_id_votes_latest.csv',
            OUTPUT_DIR / 'submission_id_votes_final_round_v2.csv',
            OUTPUT_DIR / 'submission_id_votes_targeted_plus_page1.csv',
            OUTPUT_DIR / 'submission_id_votes_refresh.csv',
        ])

    return current_ocr, current_submission


def next_batch_number() -> int:
    numbers: set[int] = set()
    candidate_paths: list[Path] = []
    candidate_paths.extend(PROJECT_DIR.glob('temp_constituency_page1_batch*'))
    candidate_paths.extend(PROJECT_DIR.glob('temp_constituency_page1_batch*_cache'))
    candidate_paths.extend(OUTPUT_DIR.glob('temp_constituency_page1_batch*_partial.csv'))
    candidate_paths.extend(OUTPUT_DIR.glob('ocr_results_final_round_plus_page1_batch*.csv'))
    candidate_paths.extend(OUTPUT_DIR.glob('submission_id_votes_batch*_auto_balance.csv'))

    for path in candidate_paths:
        match = BATCH_NUMBER_RE.search(path.name)
        if match:
            numbers.add(int(match.group(1)))

    return (max(numbers) + 1) if numbers else 2


def summarize_submission(submission_csv: Path) -> dict[str, int]:
    template_rows = {row['id']: row for row in load_csv(TEMPLATE_CSV)}
    submission_rows = load_csv(submission_csv)

    zero_blank = 0
    zero_named = 0
    for row in submission_rows:
        if row['votes'] != '0':
            continue
        template_row = template_rows[row['id']]
        if template_row['party_name'].strip():
            zero_named += 1
        else:
            zero_blank += 1

    return {
        'total_rows': len(submission_rows),
        'zero_named': zero_named,
        'zero_blank': zero_blank,
        'zero_total': zero_named + zero_blank,
    }


def remaining_constituency_docs_without_cache(submission_csv: Path) -> list[tuple[int, str]]:
    template_rows = {row['id']: row for row in load_csv(TEMPLATE_CSV)}
    submission_rows = load_csv(submission_csv)
    zero_named_docs = Counter()

    for row in submission_rows:
        template_row = template_rows[row['id']]
        if row['votes'] != '0':
            continue
        if not template_row['party_name'].strip():
            continue
        if not template_row['doc_id'].startswith('constituency_'):
            continue
        zero_named_docs[template_row['doc_id']] += 1

    cache_dirs = find_page1_cache_dirs()
    remaining: list[tuple[int, str]] = []
    for doc_id, count in zero_named_docs.items():
        has_page1_cache = any(
            (cache_dir / f'{doc_id}.md').exists() or (cache_dir / f'{doc_id}_page1.md').exists()
            for cache_dir in cache_dirs
        )
        if not has_page1_cache:
            remaining.append((count, doc_id))

    return sorted(remaining, key=lambda item: (-item[0], item[1]))


def prepare_constituency_batch(submission_csv: Path, batch_dir: Path, limit: int = 10) -> list[tuple[int, str]]:
    if batch_dir.exists():
        shutil.rmtree(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)

    selected_docs = remaining_constituency_docs_without_cache(submission_csv)[:limit]
    copied_docs: list[tuple[int, str]] = []
    for count, doc_id in selected_docs:
        image_path = IMAGES_DIR / f'{doc_id}.png'
        if image_path.exists():
            shutil.copy2(image_path, batch_dir / image_path.name)
            copied_docs.append((count, doc_id))
        else:
            print(f'[WARN] Missing image for {doc_id}: {image_path}')

    return copied_docs


def merge_ocr_csv(base_csv: Path, extra_csv: Path, merged_csv: Path) -> None:
    seen: set[tuple[str, str, str, str]] = set()
    rows: list[dict[str, str]] = []

    for path in (base_csv, extra_csv):
        if not path.exists():
            continue
        for row in load_csv(path):
            key = (row['id_doc'], row['row_num'], row['party_name'], row['vote'])
            if key in seen:
                continue
            seen.add(key)
            rows.append(row)

    with merged_csv.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['id', 'id_doc', 'row_num', 'party_name', 'vote'])
        writer.writeheader()
        for index, row in enumerate(rows, start=1):
            writer.writerow({
                'id': index,
                'id_doc': row['id_doc'],
                'row_num': row['row_num'],
                'party_name': row['party_name'],
                'vote': row['vote'],
            })


def run_full_refresh(page_mode: str = 'likely-tables') -> tuple[Path, Path]:
    assert_required_paths(require_images=True, copy_images=COPY_IMAGES_TO_LOCAL)

    base_ocr_csv = OUTPUT_DIR / 'ocr_results_refresh.csv'
    base_submission_csv = OUTPUT_DIR / 'submission_id_votes_refresh.csv'
    base_diagnostics_json = OUTPUT_DIR / 'submission_id_votes_refresh_diagnostics.json'

    run_cmd([
        PYTHON_BIN, str(OCR_SCRIPT),
        '--input-dir', str(IMAGES_DIR),
        '--cache-dir', str(PROJECT_DIR / 'ocr_cache'),
        '--output-csv', str(base_ocr_csv),
        '--page-mode', page_mode,
        '--workers', str(WORKERS),
        '--max-requests-per-minute', str(MAX_REQUESTS_PER_MINUTE),
        '--sleep-seconds', str(SLEEP_SECONDS),
        '--retries', str(RETRIES),
        '--continue-on-error',
    ])

    run_cmd([
        PYTHON_BIN, str(BUILD_SCRIPT),
        '--ocr-csv', str(base_ocr_csv),
        '--template-csv', str(TEMPLATE_CSV),
        '--output-csv', str(base_submission_csv),
        '--diagnostics-json', str(base_diagnostics_json),
    ])

    sync_runtime_back_to_drive()

    summary = summarize_submission(base_submission_csv)
    print(summary)
    return base_ocr_csv, base_submission_csv


def run_constituency_batch(limit: int = 10) -> dict[str, str] | None:
    assert_required_paths(require_images=True, copy_images=False)

    current_ocr_csv, current_submission_csv = detect_current_paths()
    if current_ocr_csv is None or current_submission_csv is None:
        raise RuntimeError('Run the baseline refresh first, or copy the latest output files into Drive.')

    batch_number = next_batch_number()
    batch_tag = f'batch{batch_number}'
    batch_dir = PROJECT_DIR / f'temp_constituency_page1_{batch_tag}'
    batch_cache_dir = PROJECT_DIR / f'temp_constituency_page1_{batch_tag}_cache'
    batch_ocr_csv = OUTPUT_DIR / f'temp_constituency_page1_{batch_tag}_partial.csv'
    merged_ocr_csv = OUTPUT_DIR / f'ocr_results_final_round_plus_page1_{batch_tag}.csv'
    submission_csv = OUTPUT_DIR / f'submission_id_votes_{batch_tag}_auto_balance.csv'
    diagnostics_json = OUTPUT_DIR / f'submission_id_votes_{batch_tag}_auto_balance_diagnostics.json'

    selected_docs = prepare_constituency_batch(current_submission_csv, batch_dir, limit=limit)
    if not selected_docs:
        print('No remaining constituency page-1 docs without cache, or matching images were not found.')
        return None

    print('Selected docs:', selected_docs)

    run_cmd([
        PYTHON_BIN, str(OCR_SCRIPT),
        '--input-dir', str(batch_dir),
        '--cache-dir', str(batch_cache_dir),
        '--output-csv', str(batch_ocr_csv),
        '--page-mode', 'all',
        '--workers', str(WORKERS),
        '--max-requests-per-minute', str(MAX_REQUESTS_PER_MINUTE),
        '--sleep-seconds', str(SLEEP_SECONDS),
        '--retries', str(RETRIES),
        '--continue-on-error',
    ])

    merge_ocr_csv(current_ocr_csv, batch_ocr_csv, merged_ocr_csv)

    run_cmd([
        PYTHON_BIN, str(BUILD_SCRIPT),
        '--ocr-csv', str(merged_ocr_csv),
        '--template-csv', str(TEMPLATE_CSV),
        '--output-csv', str(submission_csv),
        '--diagnostics-json', str(diagnostics_json),
    ])

    sync_runtime_back_to_drive()

    summary = summarize_submission(submission_csv)
    print(summary)

    return {
        'batch_tag': batch_tag,
        'current_ocr_csv': str(merged_ocr_csv),
        'current_submission_csv': str(submission_csv),
        'selected_docs': str(selected_docs),
    }


prepare_runtime(copy_images=COPY_IMAGES_TO_LOCAL)
CURRENT_OCR_CSV, CURRENT_SUBMISSION_CSV = detect_current_paths()
print('PYTHON_BIN =', PYTHON_BIN)
print('CURRENT_OCR_CSV =', CURRENT_OCR_CSV)
print('CURRENT_SUBMISSION_CSV =', CURRENT_SUBMISSION_CSV)

## Optional baseline refresh

Run this cell if you want to rebuild the baseline OCR result from scratch.

Tips:

- Leave `COPY_IMAGES_TO_LOCAL = False` for the quickest startup.
- Set `COPY_IMAGES_TO_LOCAL = True` only if you plan to do a long full refresh and Drive I/O feels slow.
- Start with `SPEED_PROFILE = 'balanced'`. If the API starts rate-limiting, switch to `conservative`.

In [ ]:
CURRENT_OCR_CSV, CURRENT_SUBMISSION_CSV = run_full_refresh(page_mode='likely-tables')
print('Baseline OCR =', CURRENT_OCR_CSV)
print('Baseline submission =', CURRENT_SUBMISSION_CSV)

## Constituency page-1 refinement

Rerun this cell multiple times.
Each run creates the next batch automatically, rebuilds the submission locally, then syncs results back to Drive.

In [ ]:
result = run_constituency_batch(limit=BATCH_LIMIT)
result

In [ ]:
CURRENT_OCR_CSV, CURRENT_SUBMISSION_CSV = detect_current_paths()
print('CURRENT_OCR_CSV =', CURRENT_OCR_CSV)
print('CURRENT_SUBMISSION_CSV =', CURRENT_SUBMISSION_CSV)
if CURRENT_SUBMISSION_CSV is None:
    raise RuntimeError('No current submission CSV found. Run the baseline cell first or copy the latest output files into Drive.')
print('SUMMARY =', summarize_submission(CURRENT_SUBMISSION_CSV))
print('REMAINING constituency docs without page1 cache =', remaining_constituency_docs_without_cache(CURRENT_SUBMISSION_CSV)[:20])
pd.read_csv(CURRENT_SUBMISSION_CSV).head()